# Check JB_prompts progress (Colab)

Downloads the latest `answercarefully_jb.sqlite3` from Google Drive and reports, for every row, whether its mixed columns are:

| Status | Meaning |
|---|---|
| **filled** | all four `mix_*` columns have a value |
| **half filled** | only the Japanese-base pair (`mix_every2_ja_base`, `mix_every3_ja_base`) or only the English-base pair is filled |
| **missing (failed)** | Qwen was asked but no reply passed the checks; `mix_error` says why. JB_prompts retries these on its next run |
| **missing (not run yet)** | outside the `LIMIT` of every run so far |

No GPU needed: any Colab server works (**Select Kernel → Colab → New Colab Server**, CPU is fine).

In [ ]:
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/ez-penta")
DB = DRIVE_DIR / "answercarefully_jb.sqlite3"
CHECKPOINT = DRIVE_DIR / "jb_prompts_checkpoint.jsonl"
WORK_DIR = Path("/content/work")

JA_BASE = ["mix_every2_ja_base", "mix_every3_ja_base"]
EN_BASE = ["mix_every2_en_base", "mix_every3_en_base"]

## 1. Download the db from Google Drive

In [ ]:
import shutil

if not DRIVE_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")
if not DB.exists():
    raise FileNotFoundError(f"{DB} not found. Run JB_prompts.ipynb through its save cell first.")

# SQLite is unreliable directly on the Drive mount, so read a local copy.
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DB = WORK_DIR / DB.name
shutil.copy(DB, LOCAL_DB)
print(f"Copied {DB} -> {LOCAL_DB}")

In [ ]:
import sqlite3

import pandas as pd

connection = sqlite3.connect(LOCAL_DB)
records = pd.read_sql_query("SELECT * FROM records", connection)
generation_info = pd.read_sql_query("SELECT * FROM generation_info", connection)
connection.close()

print(f"{len(records)} records")
generation_info

## 2. Filled vs half filled vs missing

In [ ]:
import numpy as np

ja_done = records[JA_BASE].notna().all(axis=1)
en_done = records[EN_BASE].notna().all(axis=1)
records["status"] = np.select(
    [ja_done & en_done, ja_done | en_done, records["mix_error"].notna()],
    ["filled", "half filled", "missing (failed)"],
    "missing (not run yet)",
)
records["missing_part"] = np.select([ja_done & ~en_done, en_done & ~ja_done], ["en_base", "ja_base"], "")

summary = pd.crosstab(records["status"], records["source_split"], margins=True, margins_name="total")
summary

In [ ]:
# Half filled: which pair is missing, and why.
half = records[records["status"] == "half filled"]
print(f"{len(half)} half filled rows")
half[["source_id", "source_split", "missing_part", "text", "text_en", "mix_error"]]

In [ ]:
# Missing (failed): Qwen's replies for both parts failed the checks.
failed = records[records["status"] == "missing (failed)"]
print(f"{len(failed)} failed rows")
failed[["source_id", "source_split", "text", "mix_error"]]

In [ ]:
# Missing (not run yet): JB_prompts processes rows in table order, so these are the rows past the largest LIMIT used so far.
not_run = records[records["status"] == "missing (not run yet)"]
print(f"{len(not_run)} rows not run yet", f"(table rows {not_run.index.min()}-{not_run.index.max()})" if len(not_run) else "")
not_run[["source_id", "source_split", "text"]].head()

In [ ]:
# A few filled rows, to eyeball the mixing.
pd.set_option("display.max_colwidth", None)
records.loc[records["status"] == "filled", ["text", "text_en", *JA_BASE, *EN_BASE]].head(5)

## 3. Compare with the checkpoint

The db is only rewritten when JB_prompts reaches its save cell, but the checkpoint is written after every batch. If a run was cut off, the checkpoint can be ahead of the db; rerunning JB_prompts brings the db up to date.

In [ ]:
import json

if CHECKPOINT.exists():
    latest = {}
    with CHECKPOINT.open(encoding="utf-8") as file:
        for line in file:
            record = json.loads(line)
            # Same rule as JB_prompts: the last line for a row and task wins.
            latest[(record["source_id"], record["source_split"], record["task"])] = record["result"] is not None
    done = {task: [latest.get((row.source_id, row.source_split, task), False) for row in records.itertuples(index=False)]
            for task in ("ja", "en")}
    ja_ok, en_ok = np.array(done["ja"]), np.array(done["en"])
    comparison = pd.DataFrame({
        "db": [(ja_done & en_done).sum(), (ja_done ^ en_done).sum()],
        "checkpoint": [(ja_ok & en_ok).sum(), (ja_ok ^ en_ok).sum()],
    }, index=["filled", "half filled"])
    display(comparison)
    if (comparison["db"] != comparison["checkpoint"]).any():
        print("The checkpoint is ahead of the db: rerun JB_prompts.ipynb (it skips finished rows) to save the db.")
    else:
        print("The db matches the checkpoint.")
else:
    print(f"No checkpoint at {CHECKPOINT}")